# XTTS v2 Fine-tune на голосе Алексея Андреева

**Цель**: натренировать XTTS v2 под голос Андреева, чтобы получить 85-90% сходства (вместо 60-75% zero-shot).

**Требования**:
- GPU: T4 (бесплатный Colab) или L4/A100 (Pro) — минимум 16GB VRAM
- Время: 6-12 часов
- Датасет: 107 минут (уже готов в `data/andreev_dataset/`)

## Шаги:
1. Настройка окружения
2. Загрузка датасета (из Google Drive или через wget)
3. Подготовка LJSpeech формата
4. Запуск тренировки
5. Скачивание обученной модели

## 1. Setup

In [ ]:
# Проверить что есть GPU
!nvidia-smi

In [ ]:
# Установить TTS
!pip install -q TTS==0.22.0 torch==2.1.0 torchaudio==2.1.0 transformers==4.40.0
!pip install -q huggingface_hub gradio

In [ ]:
# Монтировать Google Drive (загрузите датасет туда)
from google.colab import drive
drive.mount('/content/drive')

## 2. Загрузка датасета

**Перед запуском**: залейте папку `data/andreev_dataset/` в ваш Google Drive.

Или используйте rclone для прямой передачи.

Структура должна быть:
```
andreev_dataset/
├── segments/
│   ├── seg_0000.wav
│   ├── seg_0001.wav
│   └── ...
└── metadata.csv  # формат: "seg_XXXX.wav|текст транскрипции"
```

In [ ]:
import shutil
# Скопировать датасет из Drive в Colab (быстрее чем работать напрямую с Drive)
shutil.copytree('/content/drive/MyDrive/andreev_dataset', '/content/dataset')
!ls /content/dataset/segments | wc -l
!wc -l /content/dataset/metadata.csv

## 3. Подготовка LJSpeech формата

XTTS требует LJSpeech-like формат:
```
metadata.csv:
audio_file|text|normalized_text
```

In [ ]:
import os
import shutil

# Читаем metadata.csv (наш формат: seg_XXXX.wav|текст)
lines = []
with open('/content/dataset/metadata.csv') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            audio = parts[0]
            text = parts[1]
            # Фильтруем слишком короткие/длинные (для стабильной тренировки)
            if 1.0 < len(text) / 10 and len(text) < 200:
                lines.append(f'wavs/{audio}|{text}|{text}')

# Создаём LJSpeech-подобную структуру
os.makedirs('/content/dataset_lj/wavs', exist_ok=True)
for seg_file in os.listdir('/content/dataset/segments'):
    src = f'/content/dataset/segments/{seg_file}'
    dst = f'/content/dataset_lj/wavs/{seg_file}'
    if not os.path.exists(dst):
        shutil.copy2(src, dst)

# Записываем metadata.csv в LJSpeech формате
with open('/content/dataset_lj/metadata.csv', 'w') as f:
    f.write('\n'.join(lines))

print(f'Prepared {len(lines)} samples')
!head -5 /content/dataset_lj/metadata.csv

## 4. Запуск XTTS Fine-tune

Используем официальный recipe из Coqui TTS.

In [ ]:
# Клонируем Coqui TTS с рецептами
!git clone https://github.com/coqui-ai/TTS.git /content/coqui-tts
%cd /content/coqui-tts/recipes/ljspeech/xtts_v2/
!ls

In [ ]:
# Создаём config для нашего fine-tune
config_py = '''
import os
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig
from TTS.utils.manage import ModelManager

# Paths
OUT_PATH = "/content/drive/MyDrive/xtts_andreev_finetune"
os.makedirs(OUT_PATH, exist_ok=True)
DATASET_PATH = "/content/dataset_lj"

# Dataset config
config_dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="andreev",
    path=DATASET_PATH,
    meta_file_train="metadata.csv",
    language="ru",
)

# Pre-trained XTTS paths (downloaded automatically)
TOKENIZER_FILE = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/vocab.json"
XTTS_CHECKPOINT = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/model.pth"
XTTS_CONFIG_FILE = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/config.json"
DVAE_CHECKPOINT = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/dvae.pth"
MEL_NORM_FILE = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/mel_stats.pth"

# Speaker reference (Andreev introducing himself)
SPEAKER_REFERENCE = [f"/content/dataset_lj/wavs/seg_{i:04d}.wav" for i in [112, 151, 190, 258, 454]]

# Audio config
audio_config = XttsAudioConfig(
    sample_rate=22050,
    dvae_sample_rate=22050,
    output_sample_rate=24000,
)

# GPT config — KEY PARAMETERS
model_args = GPTArgs(
    max_conditioning_length=132300,  # 6s at 22050
    min_conditioning_length=66150,   # 3s at 22050
    debug_loading_failures=False,
    max_wav_length=255995,  # 11.6s at 22050
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

# Training config
config = GPTTrainerConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name="xtts_andreev_finetune",
    project_name="XTTS_Andreev",
    run_description="Fine-tune XTTS v2 on Andreev voice (107min telephone audio)",
    dashboard_logger="tensorboard",
    logger_uri=None,
    audio=audio_config,
    batch_size=3,          # reduce if OOM (T4 has 16GB)
    batch_group_size=48,
    eval_batch_size=3,
    num_loader_workers=8,
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=1000,
    save_step=10000,
    save_n_checkpoints=1,
    save_checkpoints=True,
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-6,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[
        {"text": "Добрый день! Компания Мультимедиа Видеосистемы, меня зовут Алексей.", "speaker_wav": SPEAKER_REFERENCE, "language": "ru"},
        {"text": "Давайте я подготовлю для вас коммерческое предложение.", "speaker_wav": SPEAKER_REFERENCE, "language": "ru"},
    ],
    epochs=10,  # start with 10, can continue
)

# Load training samples
train_samples, eval_samples = load_tts_samples(
    config_dataset,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=0.01,
)

# Init model
model = GPTTrainer.init_from_config(config)

# Init trainer
trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=True,
        grad_accum_steps=8,
    ),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()
'''

with open('/content/train_xtts.py', 'w') as f:
    f.write(config_py)

print('Training script ready at /content/train_xtts.py')

In [ ]:
# Загрузить предобученную XTTS модель
from TTS.utils.manage import ModelManager
mm = ModelManager()
mm.ask_tos = lambda *a, **k: True
mm.download_model("tts_models/multilingual/multi-dataset/xtts_v2")

In [ ]:
# Запустить тренировку (займёт 6-12 часов)
!python /content/train_xtts.py 2>&1 | tail -200

## 5. Скачать обученную модель

После тренировки файлы появятся в `/content/drive/MyDrive/xtts_andreev_finetune/`:
- `best_model.pth` — обученный чекпойнт
- `config.json` — конфиг
- `vocab.json`, `dvae.pth`, `mel_stats.pth` — из оригинальной XTTS

После скачивания заменить в проекте `models/xtts_v2/model.pth` на `best_model.pth`.

In [ ]:
# Тест обученной модели
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

_o = torch.load
def _p(*a, **k): k['weights_only']=False; return _o(*a, **k)
torch.load = _p

config = XttsConfig()
config.load_json('/content/drive/MyDrive/xtts_andreev_finetune/config.json')
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path='/content/drive/MyDrive/xtts_andreev_finetune/best_model.pth', eval=True)
model.cuda()

# Тест
ref = ['/content/dataset_lj/wavs/seg_0112.wav', '/content/dataset_lj/wavs/seg_0151.wav']
gpt_cond, spk_emb = model.get_conditioning_latents(audio_path=ref, gpt_cond_len=60)

out = model.inference(
    text='Добрый день! Компания Мультимедиа Видеосистемы, меня зовут Алексей.',
    language='ru',
    gpt_cond_latent=gpt_cond, speaker_embedding=spk_emb,
    temperature=0.55, repetition_penalty=10.0,
)

import IPython.display as ipd
ipd.Audio(out['wav'], rate=24000)